# Lab — Reusable Visual Capabilities Under One Contract

This final Beginner lab asks whether one reusable representation or promptable interface can support classification, retrieval, correspondence, grounding, and segmentation in an industrial inspection workflow.

**Success criteria:** known-answer tests pass; Factory C stays held out; prompt and vocabulary versions are explicit; global and patch evidence remain separate; oracle and end-to-end mask quality are not conflated; and every local result is labeled `foundation_model=False`.

**Safety and evidence boundary:** the corpus is procedural. The default path downloads nothing, uses no credentials, and produces no foundation-model benchmark claims. Optional official adapters are disabled and require immutable revisions, local approval, and artifact hashes.

![The Beginner sequence culminates in reusable visual foundation capabilities.](assets/beginner-track-synthesis.svg)


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import re
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not-installed"

runtime = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "torch": torch.__version__,
    "scikit_learn": package_version("scikit-learn"),
    "transformers": package_version("transformers"),
    "device": str(DEVICE),
    "seed": SEED,
}
print(json.dumps(runtime, indent=2))


## 1. Build a source-separated procedural inspection corpus

Each centered crop contains one component condition: `normal`, `scratch`, `dent`, or the newly introduced `corrosion`. Factories A, B, and C vary illumination and background. A is training, B is development, and C is a locked test source.

The crop framing is an explicit input contract. The local encoders below inspect pixels only; they never receive the hidden class label. Procedural labels are used only for evaluation and probe fitting.


In [ ]:
LABELS = ["normal", "scratch", "dent", "corrosion"]
SOURCE_STYLE = {
    "Factory A": {"background": (224, 230, 236), "part": (76, 118, 150), "split": "train"},
    "Factory B": {"background": (236, 228, 215), "part": (84, 126, 156), "split": "development"},
    "Factory C": {"background": (196, 211, 220), "part": (66, 105, 139), "split": "test"},
}

@dataclass(frozen=True)
class InspectionSample:
    sample_id: str
    source: str
    split: str
    label: str
    image: np.ndarray
    box_xyxy: tuple[int, int, int, int]
    mask: np.ndarray

def make_inspection_sample(source, label, index, seed):
    rng = np.random.default_rng(seed)
    style = SOURCE_STYLE[source]
    image = Image.new("RGB", (64, 64), style["background"])
    draw = ImageDraw.Draw(image)
    jitter_x, jitter_y = (int(v) for v in rng.integers(-2, 3, size=2))
    box = (13 + jitter_x, 13 + jitter_y, 51 + jitter_x, 51 + jitter_y)
    draw.rounded_rectangle(box, radius=7, fill=style["part"], outline=(35, 56, 72), width=2)
    cx, cy = (box[0] + box[2]) // 2, (box[1] + box[3]) // 2
    if label == "scratch":
        draw.line((box[0] + 6, box[3] - 8, box[2] - 6, box[1] + 8), fill=(250, 250, 245), width=3)
        draw.line((box[0] + 11, box[3] - 5, box[2] - 5, box[1] + 12), fill=(218, 225, 228), width=1)
    elif label == "dent":
        radius = 7 + index % 2
        draw.ellipse((cx - radius, cy - radius, cx + radius, cy + radius), fill=(28, 43, 55), outline=(18, 29, 37))
    elif label == "corrosion":
        for _ in range(9):
            x = int(rng.integers(box[0] + 5, box[2] - 4))
            y = int(rng.integers(box[1] + 5, box[3] - 4))
            r = int(rng.integers(1, 3))
            draw.ellipse((x-r, y-r, x+r, y+r), fill=(205, 92, 28))
    else:
        draw.line((box[0] + 7, cy, box[2] - 7, cy), fill=(104, 148, 176), width=2)
    array = np.asarray(image, dtype=np.uint8)
    mask = np.zeros((64, 64), dtype=bool)
    yy, xx = np.mgrid[:64, :64]
    mask[(xx >= box[0]) & (xx <= box[2]) & (yy >= box[1]) & (yy <= box[3])] = True
    return InspectionSample(f"{source[-1]}-{label[:3]}-{index:02d}", source, style["split"], label, array, box, mask)

samples = []
for source_index, source in enumerate(SOURCE_STYLE):
    for label_index, label in enumerate(LABELS):
        for index in range(10):
            samples.append(make_inspection_sample(source, label, index, 1000*source_index + 100*label_index + index))

metadata = pd.DataFrame([{"sample_id": s.sample_id, "source": s.source, "split": s.split, "label": s.label} for s in samples])
assert len(samples) == 120
assert set(metadata.loc[metadata.split == "test", "source"]) == {"Factory C"}
assert set(metadata.loc[metadata.split == "train", "source"]) == {"Factory A"}
display(metadata.groupby(["split", "source", "label"]).size().rename("count").reset_index())

fig, axes = plt.subplots(3, 4, figsize=(10, 7))
for row, source in enumerate(SOURCE_STYLE):
    for col, label in enumerate(LABELS):
        sample = next(s for s in samples if s.source == source and s.label == label)
        axes[row, col].imshow(sample.image)
        axes[row, col].set_title(f"{source[-1]} · {label}")
        axes[row, col].axis("off")
plt.suptitle("Same conditions under three source styles")
plt.tight_layout()
plt.show()


## 2. Implement the alignment primitive before an SDK

A CLIP-style system normalizes image and text embeddings, builds a scaled similarity matrix, and learns both image→text and text→image matching. The assertion below checks that aligned pairs score a lower symmetric loss than a deliberately shuffled pairing.

![Image and text encoders meet in a shared normalized space.](assets/dual-encoder-alignment.svg)


In [ ]:
def l2_normalize(tensor, eps=1e-8):
    return tensor / tensor.norm(dim=-1, keepdim=True).clamp_min(eps)

def clip_symmetric_loss(image_features, text_features, logit_scale=math.log(10.0)):
    image_features = l2_normalize(image_features)
    text_features = l2_normalize(text_features)
    logits = math.exp(float(logit_scale)) * image_features @ text_features.T
    targets = torch.arange(len(logits), device=logits.device)
    image_to_text = F.cross_entropy(logits, targets)
    text_to_image = F.cross_entropy(logits.T, targets)
    return 0.5 * (image_to_text + text_to_image), logits

base = l2_normalize(torch.randn(12, 16))
paired_text = l2_normalize(base + 0.03 * torch.randn_like(base))
matched_loss, matched_logits = clip_symmetric_loss(base, paired_text)
shuffled_loss, _ = clip_symmetric_loss(base, paired_text.roll(1, dims=0))
assert matched_logits.shape == (12, 12)
assert matched_loss < shuffled_loss
alignment_check = pd.DataFrame({"pairing": ["matched", "shuffled"], "symmetric_loss": [matched_loss.item(), shuffled_loss.item()]})
display(alignment_check)


## 3. A transparent local image–text alignment proxy

The next encoder is intentionally small and inspectable. Image evidence comes from the center crop: bright line pixels, dark dent pixels, orange corrosion pixels, color means, and edge energy. Text evidence comes from an explicit lexicon. This is useful for exercising prompt and vocabulary contracts; it is **not** CLIP, SigLIP, or a foundation model.


In [ ]:
LOCAL_ENGINE = "local_alignment_proxy"
PROXY_DISCLOSURE = {"engine": LOCAL_ENGINE, "foundation_model": False, "trained_on_foundation_corpus": False}

def aligned_image_embedding(image):
    arr = image.astype(np.float32) / 255.0
    crop = arr[10:54, 10:54]
    bright = np.mean(np.all(crop > 0.84, axis=2))
    dark = np.mean(np.mean(crop, axis=2) < 0.23)
    orange = np.mean((crop[..., 0] > 0.62) & (crop[..., 1] > 0.20) & (crop[..., 1] < 0.55) & (crop[..., 2] < 0.28))
    defect = np.array([bright / 0.05, dark / 0.12, orange / 0.06])
    defect = np.clip(defect, 0, 1.5)
    normal = max(0.05, 1.0 - float(np.max(defect)))
    gray = crop.mean(axis=2)
    edge = (np.abs(np.diff(gray, axis=0)).mean() + np.abs(np.diff(gray, axis=1)).mean())
    vector = np.array([normal, defect[0], defect[1], defect[2], crop[...,0].mean(), crop[...,1].mean(), crop[...,2].mean(), 4*edge], dtype=np.float32)
    return vector / max(np.linalg.norm(vector), 1e-8)

LEXICON = {
    "normal": np.array([1,0,0,0], dtype=float), "intact": np.array([1,0,0,0], dtype=float),
    "scratch": np.array([0,1,0,0], dtype=float), "scratched": np.array([0,1,0,0], dtype=float), "abrasion": np.array([0,0.75,0,0.25], dtype=float),
    "dent": np.array([0,0,1,0], dtype=float), "dented": np.array([0,0,1,0], dtype=float),
    "corrosion": np.array([0,0,0,1], dtype=float), "corroded": np.array([0,0,0,1], dtype=float), "rust": np.array([0,0.1,0,0.9], dtype=float),
    "wear": np.array([0,0.55,0.05,0.65], dtype=float),
}

def aligned_text_embedding(text):
    tokens = re.findall(r"[a-z]+", text.lower())
    concept = np.zeros(4, dtype=float)
    for token in tokens:
        concept += LEXICON.get(token, 0.0)
    if concept.sum() == 0:
        concept[:] = 0.25
    context = np.array([
        0.30 if "photo" in tokens else 0.0,
        0.20 if "industrial" in tokens else 0.0,
        0.16 if "close" in tokens else 0.0,
        0.10 if "surface" in tokens else 0.0,
    ])
    vector = np.concatenate([concept, context])
    return (vector / max(np.linalg.norm(vector), 1e-8)).astype(np.float32)

image_embeddings = np.stack([aligned_image_embedding(s.image) for s in samples])
assert image_embeddings.shape == (120, 8)
assert np.allclose(np.linalg.norm(image_embeddings, axis=1), 1.0)
print(PROXY_DISCLOSURE)


## 4. Prompt templates, ensembling, and vocabulary sensitivity

Zero-shot does not mean prompt-independent. We measure accuracy, macro F1, per-class recall, worst-class recall, and prediction agreement. Factory C is used for locked robustness reporting only: no template is selected, rewritten, or removed from these results. Then we add a plausible competitor—`wear`—without changing the images. That isolates candidate-vocabulary sensitivity.


In [ ]:
PROMPT_TEMPLATES = {
    "bare": "{label}",
    "photo": "a photo of {label}",
    "domain": "an industrial component with {label}",
    "close_up": "a close up photo of surface {label}",
}
PROMPT_SUITE_VERSION = "inspection-prompts-v1"
VOCABULARY_VERSION = "conditions-v1"

def zero_shot_scores(features, labels, template="{label}", temperature=0.12):
    text = np.stack([aligned_text_embedding(template.format(label=label)) for label in labels])
    similarities = features @ text.T
    logits = similarities / temperature
    probabilities = np.exp(logits - logits.max(axis=1, keepdims=True))
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    return similarities, probabilities

def classification_report_rows(y_true, y_pred, split, policy):
    return {
        "split": split, "policy": policy,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

test_idx = metadata.index[metadata.split == "test"].to_numpy()
dev_idx = metadata.index[metadata.split == "development"].to_numpy()
train_idx = metadata.index[metadata.split == "train"].to_numpy()
y = metadata.label.to_numpy()

template_rows = []
template_predictions = {}
template_recalls = {}
for name, template in PROMPT_TEMPLATES.items():
    similarities, probabilities = zero_shot_scores(image_embeddings[test_idx], LABELS, template)
    predictions = np.array(LABELS)[probabilities.argmax(axis=1)]
    template_predictions[name] = predictions
    row = classification_report_rows(y[test_idx], predictions, "Factory C", name)
    correct_col = np.array([LABELS.index(label) for label in y[test_idx]])
    recalls = precision_recall_fscore_support(y[test_idx], predictions, labels=LABELS, zero_division=0)[1]
    template_recalls[name] = recalls
    row.update({f"recall_{label}": float(value) for label, value in zip(LABELS, recalls)})
    row["worst_class_recall"] = float(recalls.min())
    row["mean_correct_similarity"] = float(similarities[np.arange(len(test_idx)), correct_col].mean())
    row["mean_margin"] = float(np.sort(probabilities, axis=1)[:, -1].mean() - np.sort(probabilities, axis=1)[:, -2].mean())
    template_rows.append(row)

template_results = pd.DataFrame(template_rows)
reference = template_predictions["bare"]
template_results["agreement_with_bare"] = [np.mean(template_predictions[name] == reference) for name in template_results.policy]
display(template_results.round(3))
prompt_recall_by_class = pd.DataFrame(template_recalls, index=LABELS)
prompt_recall_by_class["worst_template_recall"] = prompt_recall_by_class.min(axis=1)
prompt_template_reporting = {
    "evaluation_use": "Factory C reporting only; no template selection or changes",
    "selection_performed_on_test": False,
    "overall_worst_template_recall": float(prompt_recall_by_class["worst_template_recall"].min()),
}
display(prompt_recall_by_class.round(3))
print(json.dumps(prompt_template_reporting, indent=2))

ensemble_text = []
for label in LABELS:
    vectors = np.stack([aligned_text_embedding(template.format(label=label)) for template in PROMPT_TEMPLATES.values()])
    mean = vectors.mean(axis=0)
    ensemble_text.append(mean / np.linalg.norm(mean))
ensemble_text = np.stack(ensemble_text)
ensemble_sim = image_embeddings[test_idx] @ ensemble_text.T
ensemble_pred = np.array(LABELS)[ensemble_sim.argmax(axis=1)]
ensemble_result = classification_report_rows(y[test_idx], ensemble_pred, "Factory C", "prompt_ensemble")
display(pd.DataFrame([ensemble_result]).round(3))


In [ ]:
expanded_vocabulary = LABELS + ["wear"]
base_sim, base_prob = zero_shot_scores(image_embeddings[test_idx], LABELS, PROMPT_TEMPLATES["domain"])
expanded_sim, expanded_prob = zero_shot_scores(image_embeddings[test_idx], expanded_vocabulary, PROMPT_TEMPLATES["domain"])
base_pred = np.array(LABELS)[base_prob.argmax(axis=1)]
expanded_pred = np.array(expanded_vocabulary)[expanded_prob.argmax(axis=1)]

vocabulary_sensitivity = pd.DataFrame({
    "sample_id": metadata.loc[test_idx, "sample_id"].to_numpy(),
    "truth": y[test_idx], "base_prediction": base_pred, "expanded_prediction": expanded_pred,
    "changed": base_pred != expanded_pred,
    "base_max_score": base_prob.max(axis=1), "expanded_max_score": expanded_prob.max(axis=1),
})
display(vocabulary_sensitivity.groupby(["truth", "changed"]).size().rename("count").reset_index())
print("Prediction change rate:", round(vocabulary_sensitivity.changed.mean(), 3))

# Controlled boundary case: the same ambiguous feature is re-decided when `wear` enters the candidate set.
boundary_feature = aligned_text_embedding("wear")[None, :]
_, boundary_base_probability = zero_shot_scores(boundary_feature, LABELS, "{label}")
_, boundary_expanded_probability = zero_shot_scores(boundary_feature, expanded_vocabulary, "{label}")
vocabulary_boundary_example = {
    "same_feature": True,
    "base_vocabulary": LABELS,
    "base_prediction": LABELS[int(boundary_base_probability.argmax())],
    "expanded_vocabulary": expanded_vocabulary,
    "expanded_prediction": expanded_vocabulary[int(boundary_expanded_probability.argmax())],
}
assert vocabulary_boundary_example["base_prediction"] != vocabulary_boundary_example["expanded_prediction"]
print(json.dumps(vocabulary_boundary_example, indent=2))

def select_abstention_policy(features, truth):
    _, probabilities = zero_shot_scores(features, LABELS, PROMPT_TEMPLATES["domain"])
    ranked = np.sort(probabilities, axis=1)
    pred = np.array(LABELS)[probabilities.argmax(axis=1)]
    rows = []
    for minimum_score in np.linspace(0.35, 0.75, 9):
        for minimum_margin in np.linspace(0.02, 0.30, 8):
            accept = (ranked[:,-1] >= minimum_score) & ((ranked[:,-1] - ranked[:,-2]) >= minimum_margin)
            errors = np.sum(accept & (pred != truth))
            cost = errors + 0.20 * np.sum(~accept)
            rows.append((cost, minimum_score, minimum_margin, accept.mean()))
    return min(rows, key=lambda row: row[0])

policy = select_abstention_policy(image_embeddings[dev_idx], y[dev_idx])
_, test_probabilities = zero_shot_scores(image_embeddings[test_idx], LABELS, PROMPT_TEMPLATES["domain"])
test_ranked = np.sort(test_probabilities, axis=1)
test_predictions = np.array(LABELS)[test_probabilities.argmax(axis=1)]
accepted = (test_ranked[:,-1] >= policy[1]) & ((test_ranked[:,-1] - test_ranked[:,-2]) >= policy[2])
abstention_result = {
    "selected_on": "Factory B development only", "minimum_score": policy[1], "minimum_margin": policy[2],
    "test_coverage": accepted.mean(),
    "test_selective_accuracy": accuracy_score(y[test_idx][accepted], test_predictions[accepted]) if accepted.any() else None,
}
print(json.dumps(abstention_result, indent=2))


The maximum softmax value changed when the vocabulary changed, even though the image representation did not. It is therefore not a vocabulary-invariant probability. The abstention policy was selected on Factory B and then frozen for Factory C.

## 5. Representation objectives create different geometries

The following three local descriptors stand in for different representation contracts. They do not simulate the training scale or quality of ConvNeXt, DINO, or CLIP. They let us reuse Course 07's evaluation under controlled geometry.

- `specialist_proxy`: intensity/color summary optimized for a fixed crop;
- `self_supervised_proxy`: general multi-cell appearance statistics;
- `image_text_alignment_proxy`: the concept-addressable feature used above.


In [ ]:
def specialist_proxy(image):
    crop = image[10:54, 10:54].astype(np.float32) / 255.0
    gray = crop.mean(axis=2)
    return np.array([gray.mean(), gray.std(), np.quantile(gray, .1), np.quantile(gray, .5), np.quantile(gray, .9), *crop.mean(axis=(0,1))])

def self_supervised_proxy(image):
    arr = image.astype(np.float32) / 255.0
    cells = []
    for row in range(4):
        for col in range(4):
            patch = arr[row*16:(row+1)*16, col*16:(col+1)*16]
            cells.extend([patch.mean(), patch.std()])
    vector = np.asarray(cells, dtype=np.float32)
    return vector / max(np.linalg.norm(vector), 1e-8)

representations = {
    "specialist_proxy": np.stack([specialist_proxy(s.image) for s in samples]),
    "self_supervised_proxy": np.stack([self_supervised_proxy(s.image) for s in samples]),
    "image_text_alignment_proxy": image_embeddings,
}

probe_rows = []
for name, features in representations.items():
    probe = LogisticRegression(max_iter=500, random_state=SEED).fit(features[train_idx], y[train_idx])
    for split_name, indices in [("development", dev_idx), ("test", test_idx)]:
        pred = probe.predict(features[indices])
        probe_rows.append({"representation": name, **classification_report_rows(y[indices], pred, split_name, "frozen_linear_probe")})
probe_results = pd.DataFrame(probe_rows)
display(probe_results.round(3))


## 6. Retrieval and source bias

Classification probes test linear separability; retrieval tests neighborhood ordering. Protocol A uses Factory C queries against a Factory A/B gallery to measure cross-source semantic generalization. Protocol B mixes Factories A/B/C on both sides, excludes each query from its own gallery, and reports same-label@5 alongside same-source@5 and the eligible-gallery source baseline. The second protocol makes source preference measurable instead of reporting an impossible event.


In [ ]:
def row_normalize(array):
    return array / np.clip(np.linalg.norm(array, axis=1, keepdims=True), 1e-8, None)

def evaluate_retrieval(query_features, gallery_features, query_labels, gallery_labels, k=3):
    scores = row_normalize(query_features) @ row_normalize(gallery_features).T
    order = np.argsort(-scores, axis=1)
    precisions, recalls, aps = [], [], []
    for row, truth in enumerate(query_labels):
        relevant = gallery_labels == truth
        ranked = relevant[order[row]]
        precisions.append(ranked[:k].mean())
        recalls.append(ranked[:k].sum() / max(relevant.sum(), 1))
        cumulative = np.cumsum(ranked) / np.arange(1, len(ranked)+1)
        aps.append(float((cumulative * ranked).sum() / max(ranked.sum(), 1)))
    return {"P@3": np.mean(precisions), "Recall@3": np.mean(recalls), "mAP": np.mean(aps)}

def evaluate_source_bias(features, labels, sources, sample_ids, k=5):
    scores = row_normalize(features) @ row_normalize(features).T
    np.fill_diagonal(scores, -np.inf)
    order = np.argsort(-scores, axis=1)[:, :k]
    self_retrieved = sample_ids[order] == sample_ids[:, None]
    assert not self_retrieved.any(), "Each query must be excluded from its mixed-source gallery"
    same_label = labels[order] == labels[:, None]
    same_source = sources[order] == sources[:, None]
    source_baseline = np.array([(np.sum(sources == source) - 1) / (len(sources) - 1) for source in sources])
    query_rows = pd.DataFrame({
        "sample_id": sample_ids, "label": labels, "source": sources,
        "same_label@5": same_label.mean(axis=1), "same_source@5": same_source.mean(axis=1),
        "eligible_same_source_baseline": source_baseline,
    })
    summary = {
        "same_label@5": float(query_rows["same_label@5"].mean()),
        "same_source@5": float(query_rows["same_source@5"].mean()),
        "eligible_same_source_baseline": float(source_baseline.mean()),
        "source_bias_excess": float((query_rows["same_source@5"] - source_baseline).mean()),
    }
    return summary, query_rows

gallery_idx = np.concatenate([train_idx, dev_idx])
cross_source_rows, source_bias_rows = [], []
source_bias_query_tables = {}
all_sources = metadata.source.to_numpy()
all_sample_ids = metadata.sample_id.to_numpy()
for name, features in representations.items():
    cross_source_rows.append({"representation": name, "protocol": "Factory C query -> Factory A/B gallery", **evaluate_retrieval(features[test_idx], features[gallery_idx], y[test_idx], y[gallery_idx])})
    summary, query_table = evaluate_source_bias(features, y, all_sources, all_sample_ids, k=5)
    source_bias_rows.append({"representation": name, "protocol": "mixed-source self-excluded", **summary})
    source_bias_query_tables[name] = query_table
retrieval_cross_source_results = pd.DataFrame(cross_source_rows)
retrieval_source_bias_results = pd.DataFrame(source_bias_rows)
display(retrieval_cross_source_results.round(3))
display(retrieval_source_bias_results.round(3))
assert all(np.isfinite(frame["source_bias_excess"]).all() for frame in [retrieval_source_bias_results])
assert all((table["sample_id"].to_numpy() == all_sample_ids).all() for table in source_bias_query_tables.values())


## 7. Global quality does not prove patch quality

Patch correspondence asks a different question. We translate one image by exactly one 8×8 patch, compute local color/texture descriptors, and check whether nearest-neighbor matches recover that displacement for valid interior patches.

![Global and patch features support different contracts.](assets/global-patch-features.svg)


In [ ]:
def patch_features(image, patch=8):
    arr = image.astype(np.float32) / 255.0
    features, coordinates = [], []
    for row in range(0, arr.shape[0], patch):
        for col in range(0, arr.shape[1], patch):
            block = arr[row:row+patch, col:col+patch]
            gray = block.mean(axis=2)
            descriptor = [*block.mean(axis=(0,1)), *block.std(axis=(0,1)), gray.mean(), gray.std()]
            features.append(descriptor)
            coordinates.append((row // patch, col // patch))
    return row_normalize(np.asarray(features)), np.asarray(coordinates)

reference_image = next(s.image for s in samples if s.source == "Factory A" and s.label == "corrosion")
shifted_image = np.full_like(reference_image, SOURCE_STYLE["Factory A"]["background"])
shifted_image[:, 8:] = reference_image[:, :-8]
fa, ca = patch_features(reference_image)
fb, cb = patch_features(shifted_image)
nearest = (fa @ fb.T).argmax(axis=1)
matched = cb[nearest]
valid = ca[:,1] < 7
expected = ca.copy(); expected[:,1] += 1
patch_correspondence_accuracy = float(np.mean(np.all(matched[valid] == expected[valid], axis=1)))
patch_correspondence = pd.DataFrame({"query_row": ca[valid,0], "query_col": ca[valid,1], "match_row": matched[valid,0], "match_col": matched[valid,1], "correct": np.all(matched[valid] == expected[valid], axis=1)})
print("Exact one-patch displacement recovery:", round(patch_correspondence_accuracy, 3))
display(patch_correspondence.head(10))

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(reference_image); axes[0].set_title("Reference")
axes[1].imshow(shifted_image); axes[1].set_title("Shifted +8 px")
for axis in axes: axis.axis("off")
plt.tight_layout(); plt.show()


Repeated background patches are intentionally ambiguous, so a global average would hide useful and failed correspondences. Dense-feature evaluation must include spatially meaningful slices.

## 8. A transparent open-vocabulary grounding proxy

The scene contains two valves, a pipe, and a corroded connector. Proposals are created from connected non-background color regions. Region descriptors and text descriptors share explicit color and shape axes. Relational prompts add a separate pairwise stage: the proxy identifies the relation target, constructs candidate-to-target geometry, and scores `left_of`, `right_of`, distance, and vertical overlap. Hidden ground truth is never used to create or rank proposals.

This tests the interface and metric contract; it is not Grounding DINO inference.

![Open-vocabulary category detection and phrase grounding ask related but distinct questions.](assets/open-vocabulary-grounding.svg)


In [ ]:
SCENE_BG = np.array([238, 240, 242], dtype=np.uint8)
scene_image = Image.new("RGB", (96, 96), tuple(SCENE_BG))
draw = ImageDraw.Draw(scene_image)
scene_truth = {
    "red valve": {"box": (12, 18, 38, 44), "color": (196, 52, 48), "shape": "round"},
    "blue valve": {"box": (60, 18, 86, 44), "color": (48, 102, 188), "shape": "round"},
    "pipe": {"box": (42, 27, 56, 82), "color": (112, 120, 128), "shape": "long"},
    "corroded connector": {"box": (14, 62, 36, 82), "color": (205, 92, 28), "shape": "square"},
}
draw.ellipse(scene_truth["red valve"]["box"], fill=scene_truth["red valve"]["color"])
draw.ellipse(scene_truth["blue valve"]["box"], fill=scene_truth["blue valve"]["color"])
draw.rectangle(scene_truth["pipe"]["box"], fill=scene_truth["pipe"]["color"])
draw.rectangle(scene_truth["corroded connector"]["box"], fill=scene_truth["corroded connector"]["color"])
scene = np.asarray(scene_image)

def connected_components(binary):
    visited = np.zeros_like(binary, dtype=bool)
    boxes = []
    height, width = binary.shape
    for y0, x0 in zip(*np.where(binary & ~visited)):
        stack, xs, ys = [(int(y0), int(x0))], [], []
        visited[y0, x0] = True
        while stack:
            y1, x1 = stack.pop(); xs.append(x1); ys.append(y1)
            for dy, dx in ((-1,0),(1,0),(0,-1),(0,1)):
                y2, x2 = y1+dy, x1+dx
                if 0 <= y2 < height and 0 <= x2 < width and binary[y2,x2] and not visited[y2,x2]:
                    visited[y2,x2] = True; stack.append((y2,x2))
        if len(xs) >= 20:
            boxes.append((min(xs), min(ys), max(xs)+1, max(ys)+1))
    return boxes

def proposal_regions(image):
    foreground = np.any(image != SCENE_BG, axis=2)
    proposals = []
    for color in np.unique(image[foreground].reshape(-1, 3), axis=0):
        mask = np.all(image == color, axis=2)
        for box in connected_components(mask):
            proposals.append({"box": box, "mask": mask, "color": color.astype(float) / 255.0})
    return proposals

def region_descriptor(region, image_shape):
    x1,y1,x2,y2 = region["box"]; width, height = x2-x1, y2-y1
    area = region["mask"][y1:y2,x1:x2].sum(); fill = area / max(width*height, 1)
    color = region["color"]
    color_axes = [color[0]-max(color[1],color[2]), color[2]-max(color[0],color[1]), min(color[0],color[1])-color[2], 1-np.std(color)]
    shape_axes = [1.0 if 0.75 < width/height < 1.33 and fill < .9 else 0.0, 1.0 if max(width/height,height/width) > 1.7 else 0.0]
    location = [(1-(x1+x2)/(2*image_shape[1])), (x1+x2)/(2*image_shape[1])]
    vector = np.array([*color_axes, *shape_axes, *location], dtype=float)
    vector = np.clip(vector, 0, None)
    return vector / max(np.linalg.norm(vector), 1e-8)

def grounding_text_descriptor(phrase):
    tokens = set(re.findall(r"[a-z]+", phrase.lower()))
    vector = np.zeros(6, dtype=float)
    if tokens & {"red", "crimson"}: vector[0] += 1
    if tokens & {"blue", "azure"}: vector[1] += 1
    if tokens & {"corroded", "rusty", "orange"}: vector[2] += 1
    if "gray" in tokens or "pipe" in tokens: vector[3] += .8
    if tokens & {"valve", "wheel"}: vector[4] += 1
    if "pipe" in tokens: vector[5] += 1
    if "connector" in tokens: vector[2] += .8
    return vector / max(np.linalg.norm(vector), 1e-8)

RELATION_PATTERN = re.compile(r"\b(left of|right of|beside|near)\b")

def parse_grounding_prompt(prompt):
    match = RELATION_PATTERN.search(prompt.lower())
    if not match:
        return {"referent": prompt, "relation": None, "relation_target": None}
    relation = match.group(1).replace(" ", "_")
    return {"referent": prompt[:match.start()].strip(), "relation": relation, "relation_target": prompt[match.end():].strip()}

def relationship_features(candidate_box, target_box, image_shape):
    cx1,cy1,cx2,cy2 = candidate_box; tx1,ty1,tx2,ty2 = target_box
    candidate_center = np.array([(cx1+cx2)/2, (cy1+cy2)/2])
    target_center = np.array([(tx1+tx2)/2, (ty1+ty2)/2])
    horizontal_gap = max(tx1-cx2, cx1-tx2, 0)
    vertical_overlap = max(0, min(cy2,ty2)-max(cy1,ty1)) / max(min(cy2-cy1,ty2-ty1), 1)
    distance = float(np.linalg.norm(candidate_center-target_center) / np.linalg.norm(image_shape[:2]))
    return {
        "left_of": float(candidate_center[0] < target_center[0]),
        "right_of": float(candidate_center[0] > target_center[0]),
        "distance_to": distance,
        "vertical_overlap": float(vertical_overlap),
        "beside": float(horizontal_gap <= 12 and vertical_overlap >= .25),
        "near": float(distance <= .40),
    }

def build_relation_graph(proposals, image_shape):
    return {
        (candidate_index, target_index): relationship_features(candidate["box"], target["box"], image_shape)
        for candidate_index, candidate in enumerate(proposals)
        for target_index, target in enumerate(proposals)
        if candidate_index != target_index
    }

def local_grounding_proxy(image, prompt, threshold=0.60):
    parsed = parse_grounding_prompt(prompt)
    proposals = proposal_regions(image)
    relation_graph = build_relation_graph(proposals, image.shape)
    descriptors = np.stack([region_descriptor(proposal, image.shape)[:6] for proposal in proposals])
    referent_text = grounding_text_descriptor(parsed["referent"])
    referent_scores = descriptors @ referent_text
    target_index, target_semantic_score = None, None
    if parsed["relation"]:
        target_text = grounding_text_descriptor(parsed["relation_target"])
        target_scores = descriptors @ target_text
        target_index = int(np.argmax(target_scores))
        target_semantic_score = float(target_scores[target_index])
    rows = []
    for index, proposal in enumerate(proposals):
        base_score = float(referent_scores[index])
        relation_features, relation_score, target_box = None, None, None
        if target_index is not None:
            if index == target_index:
                continue
            target_box = proposals[target_index]["box"]
            relation_features = relation_graph[(index, target_index)]
            relation_score = relation_features[parsed["relation"]]
            if relation_score == 0:
                continue
            score = .75 * base_score + .25 * relation_score
        else:
            score = base_score
        if score >= threshold:
            rows.append({
                "prompt": prompt, "box": proposal["box"], "score": float(score),
                "referent_score": base_score, "relation": parsed["relation"],
                "relation_score": relation_score, "relation_target_box": target_box,
                "relation_target_semantic_score": target_semantic_score,
                "relationship_features": relation_features,
                "engine": "local_grounding_proxy", "foundation_model": False,
            })
    return sorted(rows, key=lambda row: row["score"], reverse=True)

grounding_examples = {prompt: local_grounding_proxy(scene, prompt) for prompt in ["valve", "red valve", "crimson wheel", "red valve beside pipe", "red valve right of pipe", "blue valve right of pipe", "pressure gauge"]}
display(pd.DataFrame([{"prompt": prompt, "detections": len(rows), "top_box": rows[0]["box"] if rows else None, "top_score": rows[0]["score"] if rows else None, "relation_score": rows[0]["relation_score"] if rows else None, "relation_target_box": rows[0]["relation_target_box"] if rows else None, "relationship_features": rows[0]["relationship_features"] if rows else None} for prompt, rows in grounding_examples.items()]))
grounding_relation_traces = []
for prompt, rows in grounding_examples.items():
    parsed = parse_grounding_prompt(prompt)
    if parsed["relation"]:
        grounding_relation_traces.append({"prompt":prompt, "parsed":parsed, "matched":bool(rows), "top_result":rows[0] if rows else None})
plt.figure(figsize=(4,4)); plt.imshow(scene); plt.title("Procedural grounding scene"); plt.axis("off"); plt.show()


## 9. Evaluate category, attribute, and relational grounding separately

`valve` tests category detection, `red valve` tests attribute binding, and `red valve beside pipe` tests a referent-to-target relation. The negative relation `red valve right of pipe` must return no region even though a red valve is present. An absent concept expects no regions. We match predictions to ground truth at IoU ≥ 0.5 and report precision/recall without renaming this tiny test as benchmark AP.


In [ ]:
def box_iou(first, second):
    ax1,ay1,ax2,ay2 = first; bx1,by1,bx2,by2 = second
    inter = max(0,min(ax2,bx2)-max(ax1,bx1)) * max(0,min(ay2,by2)-max(ay1,by1))
    area_a = max(0,ax2-ax1)*max(0,ay2-ay1); area_b = max(0,bx2-bx1)*max(0,by2-by1)
    return inter / max(area_a + area_b - inter, 1e-8)

GROUNDING_CASES = {
    "valve": [scene_truth["red valve"]["box"], scene_truth["blue valve"]["box"]],
    "industrial valve": [scene_truth["red valve"]["box"], scene_truth["blue valve"]["box"]],
    "red valve": [scene_truth["red valve"]["box"]],
    "crimson wheel": [scene_truth["red valve"]["box"]],
    "red valve beside pipe": [scene_truth["red valve"]["box"]],
    "red valve right of pipe": [],
    "blue valve right of pipe": [scene_truth["blue valve"]["box"]],
    "pressure gauge": [],
}
GROUNDING_CONTRACTS = {
    "valve": "category_detection", "industrial valve": "category_detection",
    "red valve": "attribute_grounding", "crimson wheel": "attribute_grounding",
    "red valve beside pipe": "relational_phrase_grounding",
    "red valve right of pipe": "relational_phrase_grounding",
    "blue valve right of pipe": "relational_phrase_grounding",
    "pressure gauge": "absent_concept",
}

def evaluate_grounding_case(predictions, targets, iou_threshold=.5):
    matched = set(); true_positive = 0
    for prediction in predictions:
        options = [(box_iou(prediction["box"], target), index) for index, target in enumerate(targets) if index not in matched]
        if options and max(options)[0] >= iou_threshold:
            _, index = max(options); matched.add(index); true_positive += 1
    false_positive = len(predictions)-true_positive; false_negative = len(targets)-true_positive
    precision = true_positive/max(len(predictions),1); recall = true_positive/max(len(targets),1) if targets else float(len(predictions)==0)
    return {"TP":true_positive,"FP":false_positive,"FN":false_negative,"precision":precision,"recall":recall}

grounding_rows = []
for prompt, targets in GROUNDING_CASES.items():
    predictions = local_grounding_proxy(scene, prompt)
    grounding_rows.append({"prompt": prompt, "contract": GROUNDING_CONTRACTS[prompt], **evaluate_grounding_case(predictions, targets)})
grounding_evaluation = pd.DataFrame(grounding_rows)
display(grounding_evaluation)
assert grounding_evaluation.loc[grounding_evaluation.prompt == "pressure gauge", "FP"].item() == 0
assert grounding_evaluation.loc[grounding_evaluation.prompt == "red valve right of pipe", "FP"].item() == 0
assert grounding_evaluation.loc[grounding_evaluation.prompt == "red valve beside pipe", "TP"].item() == 1


## 10. Compose grounding with a promptable mask proxy

The segmenter chooses the center color inside a supplied box and returns that connected color component within the box. It is an inspectable prompt protocol—not SAM. We compare oracle, perturbed, correct grounded, and wrong-instance boxes.

![Grounding and segmentation create a measurable chain of contracts.](assets/detector-segmenter-composition.svg)


In [ ]:
def truth_mask(image, box):
    x1,y1,x2,y2 = box
    color = image[(y1+y2)//2, (x1+x2)//2]
    return np.all(image == color, axis=2)

def box_prompt_segmenter_proxy(image, box):
    x1,y1,x2,y2 = [int(v) for v in box]
    x1,x2 = np.clip([x1,x2],0,image.shape[1]); y1,y2 = np.clip([y1,y2],0,image.shape[0])
    crop = image[y1:y2,x1:x2]
    if crop.size == 0:
        return np.zeros(image.shape[:2], dtype=bool)
    center_color = crop[(y2-y1)//2, (x2-x1)//2]
    selected = np.all(image == center_color, axis=2)
    window = np.zeros_like(selected); window[y1:min(y2+1,image.shape[0]),x1:min(x2+1,image.shape[1])] = True
    return selected & window

def binary_iou(prediction, target):
    intersection = np.logical_and(prediction,target).sum(); union = np.logical_or(prediction,target).sum()
    return float(intersection/union) if union else 1.0

def boundary(mask):
    padded = np.pad(mask, 1)
    interior = mask & padded[:-2,1:-1] & padded[2:,1:-1] & padded[1:-1,:-2] & padded[1:-1,2:]
    return mask & ~interior

def boundary_f1(prediction, target, tolerance=1):
    pred_boundary, target_boundary = boundary(prediction), boundary(target)
    def dilate(mask):
        padded = np.pad(mask, tolerance)
        out = np.zeros_like(mask)
        for dy in range(2*tolerance+1):
            for dx in range(2*tolerance+1): out |= padded[dy:dy+mask.shape[0], dx:dx+mask.shape[1]]
        return out
    if not pred_boundary.any() and not target_boundary.any(): return 1.0
    if not pred_boundary.any() or not target_boundary.any(): return 0.0
    precision = (pred_boundary & dilate(target_boundary)).sum()/pred_boundary.sum()
    recall = (target_boundary & dilate(pred_boundary)).sum()/target_boundary.sum()
    return float(2*precision*recall/max(precision+recall,1e-8))

red_box = scene_truth["red valve"]["box"]
red_mask = truth_mask(scene, red_box)
correct_grounded = local_grounding_proxy(scene, "red valve beside pipe")[0]["box"]
wrong_grounded = local_grounding_proxy(scene, "blue valve right of pipe")[0]["box"]
box_cases = {
    "oracle_box": red_box,
    "perturbed_box": (red_box[0]+8, red_box[1], red_box[2]-2, red_box[3]),
    "grounded_correct_instance": correct_grounded,
    "grounded_wrong_instance": wrong_grounded,
}
composition_rows = []
for case, box in box_cases.items():
    mask = box_prompt_segmenter_proxy(scene, box)
    composition_rows.append({
        "case": case, "box_IoU_to_target": box_iou(box, red_box),
        "mask_IoU_to_target": binary_iou(mask, red_mask), "boundary_F1": boundary_f1(mask, red_mask),
        "segmenter_output_nonempty": bool(mask.any()),
        "engine": "local_prompt_segmenter_proxy", "foundation_model": False,
    })
composition_evaluation = pd.DataFrame(composition_rows)
display(composition_evaluation.round(3))
assert composition_evaluation.loc[composition_evaluation.case == "oracle_box", "mask_IoU_to_target"].item() == 1.0
assert composition_evaluation.loc[composition_evaluation.case == "grounded_wrong_instance", "segmenter_output_nonempty"].item()
assert composition_evaluation.loc[composition_evaluation.case == "grounded_wrong_instance", "mask_IoU_to_target"].item() == 0.0


The wrong-instance case is important: the segmenter returns a coherent non-empty mask, yet the end-to-end result is completely wrong. Component-local “success” cannot replace system evaluation.

## 11. Adaptation ladder: zero-shot before more training

We hold out `corrosion` from the original specialist taxonomy, then compare: fixed specialist behavior, zero-shot alignment, and a frozen-feature linear probe given only three labeled corrosion development examples plus existing development labels. Factory C remains untouched.

![Adaptation increases task fit and operational burden together.](assets/adaptation-ladder.svg)


In [ ]:
known_labels = ["normal", "scratch", "dent"]
known_train = train_idx[np.isin(y[train_idx], known_labels)]
specialist_closed = LogisticRegression(max_iter=500, random_state=SEED).fit(representations["specialist_proxy"][known_train], y[known_train])
specialist_prediction = specialist_closed.predict(representations["specialist_proxy"][test_idx])

_, zero_shot_probability = zero_shot_scores(image_embeddings[test_idx], LABELS, PROMPT_TEMPLATES["domain"])
zero_shot_prediction = np.array(LABELS)[zero_shot_probability.argmax(axis=1)]

corrosion_support = dev_idx[y[dev_idx] == "corrosion"][:3]
known_support = dev_idx[np.isin(y[dev_idx], known_labels)]
probe_support = np.concatenate([known_train, known_support, corrosion_support])
few_label_probe = LogisticRegression(max_iter=500, random_state=SEED).fit(image_embeddings[probe_support], y[probe_support])
probe_prediction = few_label_probe.predict(image_embeddings[test_idx])

adaptation_rows = []
for name, prediction, trainable in [
    ("closed_specialist_no_new_class", specialist_prediction, specialist_closed.coef_.size),
    ("zero_shot_alignment", zero_shot_prediction, 0),
    ("frozen_feature_linear_probe_3_new_labels", probe_prediction, few_label_probe.coef_.size),
]:
    row = classification_report_rows(y[test_idx], prediction, "Factory C test", name)
    row.update({"trainable_parameters": int(trainable), "corrosion_recall": float(np.mean(prediction[y[test_idx]=="corrosion"] == "corrosion"))})
    adaptation_rows.append(row)
adaptation_results = pd.DataFrame(adaptation_rows)
display(adaptation_results.round(3))


The experiment does not establish a universal winner. It makes the evidence question concrete: does the new concept already exist in an addressable representation, and if not, what is the smallest justified adaptation?

## 12. Failure attribution across capabilities

We turn observed events into a stage-aware table. A source gap, changed-vocabulary decision, failed correspondence, phrase miss, or wrong-instance mask should not be filed under one undifferentiated “foundation model error.”


In [ ]:
failure_events = []
for _, row in vocabulary_sensitivity[vocabulary_sensitivity.changed].iterrows():
    failure_events.append({"stage":"alignment/vocabulary", "error_type":"candidate_vocabulary_flip", "example_id":row.sample_id, "source":"Factory C"})
for _, row in patch_correspondence[~patch_correspondence.correct].head(8).iterrows():
    failure_events.append({"stage":"patch_representation", "error_type":"correspondence_miss", "example_id":f"patch-{row.query_row}-{row.query_col}", "source":"Factory A synthetic shift"})
for _, row in grounding_evaluation.iterrows():
    if row.FP or row.FN:
        failure_events.append({"stage":"grounding", "error_type":"phrase_or_localization_error", "example_id":row.prompt, "source":"procedural scene"})
wrong_row = composition_evaluation[composition_evaluation.case == "grounded_wrong_instance"].iloc[0]
failure_events.append({"stage":"composition", "error_type":"precise_mask_wrong_instance", "example_id":"blue-valve-for-red-target", "source":"procedural scene"})

failure_table = pd.DataFrame(failure_events)
failure_taxonomy_summary = failure_table.groupby(["stage","error_type"]).size().rename("count").reset_index()
display(failure_taxonomy_summary)


## 13. Optional maintained model adapters—disabled by default

These manifests were reviewed on 2026-09-03. A full repository revision and Hugging Face model revision are recorded separately. The common `transformers` model/processor interface is shown for CLIP, SigLIP 2, DINOv2, and Grounding DINO. SAM 3.1 stays on its official gated repository path.

No optional adapter runs unless its individual environment flag is set. `trust_remote_code=False` is explicit. These adapters are smoke tests, not capability observations. Until resolved checkpoint and processor hashes are collected, every downloaded observation is machine-labeled `artifact_hash_status=not_collected` and `production_provenance_complete=False`; it cannot enter a model comparison or release decision.


In [ ]:
OPTIONAL_MODEL_MANIFESTS = {
    "clip": {
        "model_id":"openai/clip-vit-base-patch32", "model_revision":"3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268",
        "source_repository":"openai/CLIP", "source_revision":"d05afc436d78f1c48dc0dbf8e5980a9d471f35f6",
        "code_license":"MIT", "weight_license":"review model card and resolved artifacts",
    },
    "siglip2": {
        "model_id":"google/siglip2-base-patch16-224", "model_revision":"75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2",
        "source_repository":"google-research/big_vision", "source_revision":"0127fb6b337ee2a27bf4e54dea79cff176527356",
        "model_card_license":"Apache-2.0; confirm organizational approval",
    },
    "dinov2": {
        "model_id":"facebook/dinov2-small", "model_revision":"ed25f3a31f01632728cabb09d1542f84ab7b0056",
        "source_repository":"facebookresearch/dinov2", "source_revision":"7764ea0f912e53c92e82eb78a2a1631e92725fc8",
        "model_card_license":"Apache-2.0; confirm resolved artifacts",
    },
    "grounding_dino": {
        "model_id":"IDEA-Research/grounding-dino-tiny", "model_revision":"a2bb814dd30d776dcf7e30523b00659f4f141c71",
        "source_repository":"IDEA-Research/GroundingDINO", "source_revision":"856dde20aee659246248e20734ef9ba5214f5e44",
        "code_and_model_card_license":"Apache-2.0; review datasets and deployment",
    },
    "sam31": {
        "model_id":"facebook/sam3.1", "model_revision":"daa63191845a41281374e725f4c9e51c7a824460",
        "source_repository":"facebookresearch/sam3", "source_revision":"660a5e9e1b8b4c02c0ad97229b88a09a6e4ff5b7",
        "license":"custom SAM License, gated checkpoint, CUDA-oriented stack",
    },
    "dinov3_research_context": {
        "source_repository":"facebookresearch/dinov3", "source_revision":"6876159a11b4df116f30f667f8c9888617df0751",
        "license":"custom DINOv3 License; gated weights; not a default adapter",
    },
}

def sha256_file(path, chunk_size=1024*1024):
    digest = hashlib.sha256()
    with open(path,"rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""): digest.update(chunk)
    return digest.hexdigest()

def incomplete_download_provenance(processor):
    return {
        "artifact_hash_status": "not_collected",
        "checkpoint_sha256": None, "processor_config_sha256": None,
        "processor_class": type(processor).__name__,
        "library_versions": {"transformers": runtime["transformers"], "torch": runtime["torch"]},
        "production_provenance_complete": False,
        "comparison_eligible": False,
    }

optional_downloaded_model_observations = []

if os.getenv("CV_ENABLE_CLIP", "0") == "1":
    from transformers import AutoProcessor, CLIPModel
    manifest = OPTIONAL_MODEL_MANIFESTS["clip"]
    processor = AutoProcessor.from_pretrained(manifest["model_id"], revision=manifest["model_revision"], trust_remote_code=False)
    model = CLIPModel.from_pretrained(manifest["model_id"], revision=manifest["model_revision"], trust_remote_code=False).eval().to(DEVICE)
    inputs = processor(text=["an intact component", "a corroded component"], images=Image.fromarray(samples[test_idx[0]].image), return_tensors="pt", padding=True).to(DEVICE)
    with torch.inference_mode(): output = model(**inputs)
    optional_downloaded_model_observations.append({"adapter":"clip_transformers", "manifest":manifest, "provenance":incomplete_download_provenance(processor), "status":"smoke_test_executed; logits are not a capability evaluation", "logits_per_image":output.logits_per_image.cpu().tolist()})

if os.getenv("CV_ENABLE_SIGLIP2", "0") == "1":
    from transformers import AutoModel, AutoProcessor
    manifest = OPTIONAL_MODEL_MANIFESTS["siglip2"]
    processor = AutoProcessor.from_pretrained(manifest["model_id"], revision=manifest["model_revision"], trust_remote_code=False)
    model = AutoModel.from_pretrained(manifest["model_id"], revision=manifest["model_revision"], trust_remote_code=False).eval().to(DEVICE)
    optional_downloaded_model_observations.append({"adapter":"siglip2_transformers", "manifest":manifest, "provenance":incomplete_download_provenance(processor), "status":"smoke_test_loaded_only; no model capability was evaluated"})

if os.getenv("CV_ENABLE_DINOV2", "0") == "1":
    from transformers import AutoImageProcessor, AutoModel
    manifest = OPTIONAL_MODEL_MANIFESTS["dinov2"]
    processor = AutoImageProcessor.from_pretrained(manifest["model_id"], revision=manifest["model_revision"], trust_remote_code=False)
    model = AutoModel.from_pretrained(manifest["model_id"], revision=manifest["model_revision"], trust_remote_code=False).eval().to(DEVICE)
    inputs = processor(images=Image.fromarray(samples[test_idx[0]].image), return_tensors="pt").to(DEVICE)
    with torch.inference_mode(): output = model(**inputs)
    optional_downloaded_model_observations.append({"adapter":"dinov2_transformers", "manifest":manifest, "provenance":incomplete_download_provenance(processor), "status":"smoke_test_executed; tensor shapes are not a capability evaluation", "global_shape":list(output.last_hidden_state[:,0].shape), "patch_shape":list(output.last_hidden_state[:,1:].shape)})

if os.getenv("CV_ENABLE_GROUNDING_DINO", "0") == "1":
    from transformers import AutoProcessor, GroundingDinoForObjectDetection
    manifest = OPTIONAL_MODEL_MANIFESTS["grounding_dino"]
    processor = AutoProcessor.from_pretrained(manifest["model_id"], revision=manifest["model_revision"], trust_remote_code=False)
    model = GroundingDinoForObjectDetection.from_pretrained(manifest["model_id"], revision=manifest["model_revision"], trust_remote_code=False).eval().to(DEVICE)
    inputs = processor(images=Image.fromarray(scene), text="red valve.", return_tensors="pt").to(DEVICE)
    with torch.inference_mode(): output = model(**inputs)
    optional_downloaded_model_observations.append({"adapter":"grounding_dino_transformers", "manifest":manifest, "provenance":incomplete_download_provenance(processor), "status":"smoke_test_executed; no post-processed capability evaluation"})

if os.getenv("CV_ENABLE_SAM31", "0") == "1":
    checkpoint = Path(os.environ["SAM31_CHECKPOINT"])
    assert checkpoint.is_file(), "Provide an approved local checkpoint; implicit download is disabled"
    assert os.environ.get("SAM31_REPO_REVISION") == OPTIONAL_MODEL_MANIFESTS["sam31"]["source_revision"]
    optional_downloaded_model_observations.append({"adapter":"sam31_official_local", "manifest":OPTIONAL_MODEL_MANIFESTS["sam31"], "checkpoint_sha256":sha256_file(checkpoint), "provenance":{"artifact_hash_status":"checkpoint_collected_processor_not_collected", "processor_config_sha256":None, "production_provenance_complete":False, "comparison_eligible":False}, "status":"approved local checkpoint declared; processor integration remains incomplete"})

optional_status = {name: os.getenv(name, "0") == "1" for name in ["CV_ENABLE_CLIP","CV_ENABLE_SIGLIP2","CV_ENABLE_DINOV2","CV_ENABLE_GROUNDING_DINO","CV_ENABLE_SAM31"]}
assert all(not row["provenance"]["production_provenance_complete"] for row in optional_downloaded_model_observations)
print(json.dumps({"flags":optional_status,"observation_count":len(optional_downloaded_model_observations)}, indent=2))


## 14. Enterprise evidence and decision record

The artifact separates:

1. `locally_measured_evidence`—only values computed in this run;
2. `optional_downloaded_model_observations`—empty unless explicitly enabled; and
3. `unresolved_production_assumptions`—claims this notebook cannot establish.

![A foundation system needs separate evaluation contracts before a bounded decision.](assets/evaluation-contract.svg)


In [ ]:
def records(frame):
    return json.loads(frame.to_json(orient="records"))

decision_options = pd.DataFrame([
    {"option":"keep specialist", "best_when":"stable taxonomy and tight latency", "required_evidence":"locked-class quality, target hardware, retraining plan"},
    {"option":"zero-shot foundation interface", "best_when":"rapid taxonomy exploration", "required_evidence":"prompt/vocabulary/source robustness and abstention"},
    {"option":"frozen features + head", "best_when":"few labels and reusable geometry", "required_evidence":"probe, drift, dense/global fit, artifact versioning"},
    {"option":"adapter / partial tune", "best_when":"frozen evidence is insufficient", "required_evidence":"multi-seed gain, drift, export, base-adapter compatibility"},
    {"option":"compose detector + segmenter", "best_when":"text-to-mask workflow is required", "required_evidence":"stage and end-to-end quality, latency, review effort"},
    {"option":"collect labels first", "best_when":"concept or ground truth is ambiguous", "required_evidence":"annotation protocol, agreement, information value"},
])
display(decision_options)

locally_measured_evidence = {
    "disclosure": PROXY_DISCLOSURE,
    "dataset": {"samples":len(samples),"sources":list(SOURCE_STYLE),"splits":metadata.groupby("split").size().to_dict(),"test_source":"Factory C"},
    "alignment_known_answer": records(alignment_check),
    "prompt_template_results": records(template_results),
    "prompt_recall_by_class": records(prompt_recall_by_class.reset_index(names="label")),
    "prompt_template_reporting_policy": prompt_template_reporting,
    "prompt_ensemble_result": ensemble_result,
    "vocabulary_change_rate": float(vocabulary_sensitivity.changed.mean()),
    "vocabulary_boundary_example": vocabulary_boundary_example,
    "abstention_policy": abstention_result,
    "frozen_probe_results": records(probe_results),
    "retrieval_cross_source_results": records(retrieval_cross_source_results),
    "retrieval_source_bias_results": records(retrieval_source_bias_results),
    "patch_correspondence_accuracy": patch_correspondence_accuracy,
    "grounding_evaluation": records(grounding_evaluation),
    "grounding_relation_traces": grounding_relation_traces,
    "composition_evaluation": records(composition_evaluation),
    "adaptation_results": records(adaptation_results),
    "failure_taxonomy": records(failure_taxonomy_summary),
}

unresolved_production_assumptions = [
    "Procedural images do not establish performance on real industrial cameras or rare defects.",
    "Local alignment, grounding, and segmentation proxies are not foundation-model quality evidence.",
    "No target-hardware latency, memory, throughput, concurrency, energy, or cost SLO was measured.",
    "No official model data-provenance, license, privacy, security, or commercial-use approval is implied.",
    "Prompt suites, vocabularies, taxonomies, calibration, and human-review costs need domain validation.",
    "Optional checkpoints need resolved file hashes and repeatable preprocessing before comparison.",
]

evidence = {
    "schema_version":"course-09-foundation-vision-evidence-v1",
    "reviewed_on":"2026-09-03",
    "runtime":runtime,
    "prompt_suite_version":PROMPT_SUITE_VERSION,
    "candidate_vocabulary_version":VOCABULARY_VERSION,
    "locally_measured_evidence":locally_measured_evidence,
    "optional_model_manifests":OPTIONAL_MODEL_MANIFESTS,
    "optional_downloaded_model_observations":optional_downloaded_model_observations,
    "unresolved_production_assumptions":unresolved_production_assumptions,
    "decision_options":records(decision_options),
    "DEMONSTRATION_THRESHOLD_NOTICE":"All thresholds are teaching defaults for this procedural notebook runtime only; they are not production targets.",
}
evidence_path = ARTIFACT_DIR / "course-09-foundation-vision-evidence.json"
evidence_path.write_text(json.dumps(evidence, indent=2), encoding="utf-8")
print(f"Wrote {evidence_path} ({evidence_path.stat().st_size:,} bytes)")


## 15. Production upgrade path

| Notebook boundary | Production upgrade |
| --- | --- |
| centered procedural crop | governed capture contract, consent/provenance, duplicate and source audits |
| local feature proxies | pinned official checkpoints and processors under isolated serving environments |
| tiny prompt suite | approved multilingual synonyms, absent concepts, attributes, relations, and adversarial cases |
| in-memory inference | authenticated service, tenant isolation, batching, caching, quotas, and target-hardware SLOs |
| local thresholds | development-selected, calibrated, slice-aware release gates with rollback |
| one scene | versioned evaluation corpus with classification, retrieval, grounding, mask, and composition suites |
| JSON file | signed artifact registry, lineage, approvals, retention, monitoring, and revocation |
| manual review note | staffed queue, correction-effort metric, escalation policy, and feedback adjudication |

The governance unit is not only a checkpoint. It is a bundle of model, processor, code, prompts, vocabulary, thresholds, data, hardware, and operating policy.

## 16. Exercises

1. Add `abrasion` and `rust` as approved synonyms. Measure mean and worst-prompt recall.
2. Add an absent visual concept and tune an abstention policy on Factory B only.
3. Implement text-to-image retrieval with the same aligned proxy and compare relevance contracts.
4. Improve patch descriptors without using hidden labels; report correspondence and runtime.
5. Add a relation phrase that fails and attribute whether parsing, region evidence, or binding caused it.
6. Perturb detector boxes across a grid and plot downstream mask IoU.
7. Specify a fair comparison between an automatic specialist segmenter and a box-prompted foundation segmenter.
8. Draft a release manifest and rollback plan for one optional checkpoint.

## 17. What you should now be able to explain without code

- Why is a foundation model more than a large checkpoint?
- What becomes reusable, and what stays task-specific?
- Why is zero-shot classification still dependent on prompts and competing labels?
- Why are open-vocabulary and open-world recognition different?
- Why can global representation quality disagree with patch correspondence?
- How do category detection and phrase grounding differ?
- Why must oracle-box segmentation stay separate from end-to-end performance?
- How can a precise component output still be a system failure?
- When should a specialist beat a foundation system operationally?
- Why do foundation models still require task-specific evaluation and governance?

Beginner ends here: you can now connect pixels, encoders, tokens, objectives, boxes, masks, embeddings, temporal state, prompts, and reusable capabilities without surrendering the contracts that make the system measurable.
